In [1]:
# reddit_sentiment_praw.py
# PRAW-based Reddit sentiment analysis for portfolio tickers

import os, re, time, datetime as dt
from typing import List, Dict, Tuple, Optional
import pandas as pd
import numpy as np
import json, pathlib
import praw
from praw.exceptions import PRAWException

# -----------------------------
# 0) CONFIG
# -----------------------------

# Reddit API Credentials (you'll need to fill these in)
REDDIT_CONFIG = {
    "client_id": "YOUR_CLIENT_ID_HERE",        # From Reddit app
    "client_secret": "YOUR_CLIENT_SECRET_HERE", # From Reddit app  
    "user_agent": "financial_sentiment_analysis_v1.0_by_YourUsername"
}

# Date range for analysis (PRAW works better with recent data)
DATE_FROM = "2015-01-01"  # PRAW is better for recent data
DATE_TO   = "2015-03-31"

# Cache directory
CACHE_DIR = pathlib.Path("data/cache_praw")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Subreddit configuration - focused on active financial communities
CORE_TIER_A = ["SecurityAnalysis", "investing", "stocks", "ValueInvesting"]
CORE_TIER_B = ["wallstreetbets", "StockMarket", "options", "dividends"]

# Sector-specific subreddits
SECTOR_MAP = {
    # Tech/AI stocks
    "NVDA": ["nvidia", "MachineLearning", "artificial", "singularity"],
    "AMD":  ["AMD", "pcmasterrace", "buildapc"],
    "MSFT": ["microsoft", "azure", "dotnet"],
    "GOOGL": ["google", "MachineLearning", "selfdriving"],
    "AI": ["MachineLearning", "artificial"],
    
    # Semiconductors
    "ASML": ["semiconductors", "investing"],
    "MU": ["semiconductors", "DataHoarder"],
    "MRVL": ["semiconductors", "networking"],
    
    # Quantum/Emerging Tech
    "IONQ": ["QuantumComputing", "quantum"],
    "RGTI": ["QuantumComputing", "quantum"], 
    "QBTS": ["QuantumComputing", "quantum"],
    "PLTR": ["SecurityAnalysis", "investing"],
    
    # Energy/Clean Tech
    "PLUG": ["energy", "CleanEnergy", "renewableenergy"],
    
    # Other sectors
    "AEM": ["Gold", "mining", "investing"],
    "VERU": ["biotech", "investing"],
    "APP": ["gamedev", "advertising", "marketing"],
    "ARBE": ["SelfDrivingCars", "investing"],
    "INGM": ["biotech", "investing"],
    "RDDT": ["reddit", "IPO", "investing"],
    
    # ETFs
    "QQQ": ["ETFs", "investing"],
    "SPY": ["ETFs", "investing", "SecurityAnalysis"]
}

# Subreddit weights
SUB_WEIGHTS = {
    **{s: 1.0 for s in CORE_TIER_A},
    **{s: 0.6 for s in CORE_TIER_B},
    "_sector_default": 0.8,
    "wallstreetbets": 0.4,  # Lower weight due to noise
}

# Search limits per subreddit (PRAW has different limits than Pushshift)
SEARCH_LIMITS = {
    "core_tierA": 100,
    "core_tierB": 50, 
    "sector": 25
}

# Ticker patterns (simplified for PRAW search)
TICKER_PATTERNS = {
    "RDDT": ["reddit", "$RDDT", "RDDT"],
    "NVDA": ["nvidia", "$NVDA", "NVDA", "jensen huang"],
    "AMD": ["AMD", "$AMD", "ryzen", "epyc"],
    "ASML": ["ASML", "$ASML"],
    "MU": ["$MU", "micron technology", "micron"],
    "MRVL": ["$MRVL", "marvell"],
    "MSFT": ["microsoft", "$MSFT", "MSFT", "azure"],
    "GOOGL": ["google", "$GOOGL", "alphabet", "GOOGL"],
    "AI": ["$AI", "c3.ai", "c3 ai"],
    "IONQ": ["ionq", "$IONQ", "IONQ"],
    "RGTI": ["rigetti", "$RGTI", "RGTI"],
    "QBTS": ["d-wave", "$QBTS", "QBTS"],
    "PLUG": ["$PLUG", "plug power"],
    "AEM": ["$AEM", "agnico eagle"],
    "VERU": ["veru", "$VERU"],
    "APP": ["applovin", "$APP"],
    "ARBE": ["arbe", "$ARBE"],
    "INGM": ["inogen", "$INGM"],
    "QQQ": ["$QQQ", "QQQ", "nasdaq"],
    "SPY": ["$SPY", "SPY", "s&p 500"],
    "PLTR": ["palantir", "$PLTR", "PLTR"]
}

# Output files
RAW_OUT = "reddit_praw_raw.csv"
MONTHLY_OUT = "reddit_praw_monthly.csv"

print("PRAW configuration loaded successfully")
print(f"Date range: {DATE_FROM} to {DATE_TO}")

PRAW configuration loaded successfully
Date range: 2015-01-01 to 2015-03-31


In [2]:
# -----------------------------
# 1) PRAW Setup and Utilities  
# -----------------------------

def setup_reddit():
    """Initialize Reddit API connection"""
    try:
        reddit = praw.Reddit(
            client_id="pLqfk1M1ymfj3ih1NrVFlA",
            client_secret="_hl1434FeTi9kgv_GXAi5tBLoCaLIQ",
            user_agent="SentimentAnalysisBot"
        )
        
        # Test the connection
        print(f"Connected to Reddit API as: {reddit.user.me() if reddit.user.me() else 'Anonymous'}")
        return reddit
        
    except Exception as e:
        print(f"Failed to connect to Reddit API: {e}")
        print("Please check your Reddit API credentials in REDDIT_CONFIG")
        return None

def expand_subs_for_ticker(ticker: str) -> List[str]:
    """Get all relevant subreddits for a ticker"""
    subs = set(CORE_TIER_A + CORE_TIER_B)
    for s in SECTOR_MAP.get(ticker, []):
        subs.add(s)
    return sorted(subs)

def sub_weight(name: str) -> float:
    """Get weight for subreddit"""
    return SUB_WEIGHTS.get(name, SUB_WEIGHTS.get("_sector_default", 0.8))

def tier_of_sub(sub: str) -> str:
    """Classify subreddit tier"""
    if sub in CORE_TIER_A: return "core_tierA"
    if sub in CORE_TIER_B: return "core_tierB" 
    return "sector"

def get_search_limit(sub: str) -> int:
    """Get search limit for subreddit tier"""
    tier = tier_of_sub(sub)
    return SEARCH_LIMITS.get(tier, 25)

def is_relevant_post(title: str, selftext: str, ticker: str) -> bool:
    """Check if post is relevant to ticker using pattern matching"""
    text = f"{title} {selftext}".lower()
    patterns = TICKER_PATTERNS.get(ticker, [ticker.lower()])
    
    for pattern in patterns:
        if pattern.lower() in text:
            return True
    return False

def ok_text(text: str) -> bool:
    """Check if text is valid"""
    if not text or text.strip() == "":
        return False
    text_lower = text.lower().strip()
    if text_lower in ["[removed]", "[deleted]", ""]:
        return False
    return True

def weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    """Calculate weighted mean"""
    w = np.asarray(weights, float)
    v = np.asarray(values, float)
    sw = w.sum()
    if sw <= 0: 
        return float(np.nan)
    return float(np.dot(v, w) / sw)

print("PRAW utilities defined successfully")

PRAW utilities defined successfully


In [3]:
# -----------------------------
# 1) Utilities
# -----------------------------

def months_between(start: str, end: str) -> List[Tuple[int,int]]:
    start_dt = pd.Timestamp(start).normalize().replace(day=1)
    end_dt   = pd.Timestamp(end).normalize().replace(day=1)
    months = []
    cur = start_dt
    while cur <= end_dt:
        months.append((cur.year, cur.month))
        cur = (cur + pd.offsets.MonthBegin(1))
    return months

def expand_subs_for_ticker(ticker: str) -> List[str]:
    subs = set(CORE_TIER_A + CORE_TIER_B)
    for s in SECTOR_MAP.get(ticker, []):
        subs.add(s)
    return sorted(subs)

def sub_weight(name: str) -> float:
    return SUB_WEIGHTS.get(name, SUB_WEIGHTS.get("_sector_default", 0.8))

def compile_regex(pat: str):
    return re.compile(pat, flags=re.IGNORECASE)

def ok_text(text: str) -> bool:
    if text is None: return False
    t = text.strip().lower()
    if t in ("[removed]","[deleted]",""): return False
    return True

def tier_of_sub(sub: str) -> str:
    if sub in CORE_TIER_A: return "core_tierA"
    if sub in CORE_TIER_B: return "retail"
    return "sector"

def weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    w = np.asarray(weights, float)
    v = np.asarray(values, float)
    sw = w.sum()
    if sw <= 0: return float(np.nan)
    return float(np.dot(v, w) / sw)

print("Utility functions defined successfully")

Utility functions defined successfully


In [4]:
# -----------------------------
# 2) Caching and API Functions
# -----------------------------

def cache_key(params: dict) -> pathlib.Path:
    s = json.dumps(params, sort_keys=True)
    h = hashlib.md5(s.encode()).hexdigest()
    return CACHE_DIR / f"{h}.json"

def cached_pushshift(params: dict) -> list:
    p = cache_key(params)
    if p.exists():
        try:
            return json.loads(p.read_text())
        except Exception:
            pass
    r = requests.get(PUSHSHIFT, params=params, timeout=30)
    r.raise_for_status()
    data = r.json().get("data", [])
    try:
        p.write_text(json.dumps(data))
    except Exception:
        pass
    return data

def fetch_pushshift_month(subreddit: str, query: str, year: int, month: int, tier_cap: int) -> list:
    start = pd.Timestamp(year=year, month=month, day=1, tz='UTC')
    end = (start + pd.offsets.MonthBegin(1))
    params = {
        "subreddit": subreddit,
        "q": query,
        "after": int(start.timestamp()),
        "before": int(end.timestamp()),
        "size": 500,             # max allowed; usually 1 call/month/sub
        "sort": "desc",
        "sort_type": "score"
    }
    data = cached_pushshift(params)
    # Hard cap within the month to limit volume
    return data[:tier_cap]

def finbert_pipeline():

    from transformers import AutoTokenizer, AutoModelForSequenceClassification, TextClassificationPipeline
    import torch
    model_name = "ProsusAI/finbert"
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModelForSequenceClassification.from_pretrained(model_name)
    pipe = TextClassificationPipeline(
        model=mdl, tokenizer=tok, return_all_scores=True, truncation=True, max_length=256, device=0 if torch.cuda.is_available() else -1
    )
    return ("finbert", pipe)


def score_text(analyzer, mode: str, text: str) -> Tuple[float,float,float,float]:
    """
    returns: (tone, p_pos, p_neu, p_neg)
    tone = p_pos - p_neg
    """
    text = text[:1000]  # guardrail
    if mode == "finbert":
        outs = analyzer(text)
        # outs = [[{'label': 'positive','score':..}, {'label': 'negative',...}, {'label':'neutral',...}]]
        scores = {d['label'].lower(): d['score'] for d in outs[0]}
        p_pos = float(scores.get("positive", 0.0))
        p_neg = float(scores.get("negative", 0.0))
        p_neu = float(scores.get("neutral", 0.0))
        tone = p_pos - p_neg
        return tone, p_pos, p_neu, p_neg
    else:  # vader
        s = analyzer.polarity_scores(text)
        # map to pseudo-probs (approximate)
        comp = s['compound']
        p_pos = s['pos']; p_neg = s['neg']; p_neu = s['neu']
        tone = p_pos - p_neg if (p_pos+p_neg) > 0 else comp
        return tone, p_pos, p_neu, p_neg

print("Caching and API functions defined successfully")

Caching and API functions defined successfully


In [5]:
# -----------------------------
# 2) PRAW Data Collection
# -----------------------------

def fetch_subreddit_posts(reddit, subreddit_name: str, ticker: str, limit: int = 100) -> List[Dict]:
    """Fetch posts from a subreddit using multiple search strategies"""
    posts = []
    
    try:
        subreddit = reddit.subreddit(subreddit_name)
        search_terms = TICKER_PATTERNS.get(ticker, [ticker])
        
        # Try multiple search terms
        for term in search_terms[:3]:  # Limit to avoid too many API calls
            try:
                # Search recent posts
                for submission in subreddit.search(term, limit=limit//len(search_terms), sort='new', time_filter='year'):
                    if is_relevant_post(submission.title, submission.selftext, ticker):
                        posts.append({
                            'id': submission.id,
                            'title': submission.title,
                            'selftext': submission.selftext,
                            'url': submission.url,
                            'score': submission.score,
                            'num_comments': submission.num_comments,
                            'created_utc': submission.created_utc,
                            'subreddit': subreddit_name,
                            'ticker': ticker
                        })
                
                # Small delay to be nice to API
                time.sleep(0.5)
                
            except Exception as e:
                print(f"Search failed for {term} in {subreddit_name}: {e}")
                continue
        
        # Remove duplicates by ID
        seen_ids = set()
        unique_posts = []
        for post in posts:
            if post['id'] not in seen_ids:
                seen_ids.add(post['id'])
                unique_posts.append(post)
        
        print(f"Found {len(unique_posts)} relevant posts for {ticker} in r/{subreddit_name}")
        return unique_posts
        
    except Exception as e:
        print(f"Error accessing r/{subreddit_name}: {e}")
        return []

def collect_ticker_data(reddit, ticker: str) -> List[Dict]:
    """Collect all Reddit data for a specific ticker"""
    all_posts = []
    subreddits = expand_subs_for_ticker(ticker)
    
    print(f"\n=== Collecting data for {ticker} from {len(subreddits)} subreddits ===")
    
    for sub in subreddits:
        limit = get_search_limit(sub)
        posts = fetch_subreddit_posts(reddit, sub, ticker, limit)
        all_posts.extend(posts)
        
        # Rate limiting
        time.sleep(1)
    
    print(f"Total posts collected for {ticker}: {len(all_posts)}")
    return all_posts

print("Data collection functions defined successfully")

Data collection functions defined successfully


In [6]:
# -----------------------------
# 3) Sentiment Analysis
# -----------------------------

def setup_sentiment_analyzer():
    """Setup sentiment analysis pipeline"""

    from transformers import AutoTokenizer, AutoModelForSequenceClassification, TextClassificationPipeline
    import torch
    
    model_name = "ProsusAI/finbert"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    
    pipe = TextClassificationPipeline(
        model=model, 
        tokenizer=tokenizer, 
        top_k=None, 
        truncation=True, 
        max_length=512,
        device=0 if torch.cuda.is_available() else -1
    )
    
    return ("finbert", pipe)


def analyze_sentiment(analyzer, mode: str, text: str) -> Tuple[float, float, float, float]:
    """
    Analyze sentiment of text
    Returns: (tone, positive, neutral, negative)
    """
    if not text or len(text.strip()) < 10:
        return 0.0, 0.33, 0.34, 0.33
    
    # Truncate very long text
    text = text[:2000]
    
    try:
        if mode == "finbert":
            results = analyzer(text)
            scores = {item['label'].lower(): item['score'] for item in results}
            
            pos = scores.get('positive', 0.0)
            neg = scores.get('negative', 0.0) 
            neu = scores.get('neutral', 0.0)
            tone = pos - neg
            
            return tone, pos, neu, neg
            
        else:  # vader
            scores = analyzer.polarity_scores(text)
            tone = scores['compound']
            pos = scores['pos']
            neg = scores['neg'] 
            neu = scores['neu']
            
            return tone, pos, neu, neg
            
    except Exception as e:
        print(f"Sentiment analysis failed: {e}")
        return 0.0, 0.33, 0.34, 0.33

def process_posts_sentiment(posts: List[Dict], analyzer, mode: str) -> pd.DataFrame:
    """Process posts and add sentiment analysis"""
    processed_posts = []
    
    for post in posts:
        # Combine title and selftext
        text = f"{post['title']}\n\n{post.get('selftext', '')}"
        
        if not ok_text(text):
            continue
            
        # Analyze sentiment
        tone, pos, neu, neg = analyze_sentiment(analyzer, mode, text)
        
        # Calculate post weight based on engagement
        engagement_weight = np.log1p(post['score']) * np.log1p(post['num_comments'])
        sub_weight_val = sub_weight(post['subreddit'])
        final_weight = engagement_weight * sub_weight_val
        
        processed_posts.append({
            'ticker': post['ticker'],
            'subreddit': post['subreddit'], 
            'post_id': post['id'],
            'created_utc': pd.to_datetime(post['created_utc'], unit='s', utc=True),
            'title': post['title'][:500],  # Truncate for storage
            'url': post['url'],
            'score': post['score'],
            'num_comments': post['num_comments'],
            'tone': tone,
            'positive': pos,
            'neutral': neu, 
            'negative': neg,
            'weight': final_weight
        })
    
    return pd.DataFrame(processed_posts)

print("Sentiment analysis functions defined successfully")

Sentiment analysis functions defined successfully


In [7]:
# -----------------------------
# 4) Main Processing Pipeline
# -----------------------------

def run_praw_analysis(tickers: List[str]):
    """Main function to run Reddit sentiment analysis using PRAW"""
    
    # Setup
    reddit = setup_reddit()
    if not reddit:
        print("Failed to setup Reddit API connection")
        return None, None
    
    mode, analyzer = setup_sentiment_analyzer()
    print(f"Using sentiment analyzer: {mode}")
    
    # Collect data for all tickers
    all_posts = []
    for ticker in tickers:
        posts = collect_ticker_data(reddit, ticker)
        all_posts.extend(posts)
        
        # Rate limiting between tickers
        time.sleep(2)
    
    if not all_posts:
        print("No posts collected")
        return None, None
    
    print(f"\nTotal posts collected: {len(all_posts)}")
    
    # Process sentiment
    print("Processing sentiment analysis...")
    df = process_posts_sentiment(all_posts, analyzer, mode)
    
    if df.empty:
        print("No valid posts after processing")
        return None, None
    
    # Add date columns
    df['year'] = df['created_utc'].dt.year
    df['month'] = df['created_utc'].dt.month
    df['year_month'] = df['created_utc'].dt.to_period('M').dt.to_timestamp()
    
    # Remove duplicates
    df = df.drop_duplicates(subset=['ticker', 'post_id'])
    
    # Monthly aggregation
    monthly_data = []
    
    for (ticker, year, month), group in df.groupby(['ticker', 'year', 'month']):
        if len(group) < 5:  # Minimum posts threshold
            continue
            
        weights = group['weight'].fillna(1.0)
        if weights.sum() == 0:
            weights = pd.Series([1.0] * len(group))
        
        monthly_data.append({
            'ticker': ticker,
            'year': int(year),
            'month': int(month), 
            'article_count': len(group),
            'tone_mean': weighted_mean(group['tone'], weights),
            'positive_mean': weighted_mean(group['positive'], weights),
            'negative_mean': weighted_mean(group['negative'], weights),
            'avg_score': group['score'].mean(),
            'total_comments': group['num_comments'].sum()
        })
    
    monthly_df = pd.DataFrame(monthly_data)
    
    # Save results
    os.makedirs("data", exist_ok=True)
    df.to_csv(f"data/{RAW_OUT}", index=False)
    monthly_df.to_csv(f"data/{MONTHLY_OUT}", index=False)
    
    print(f"\nResults saved:")
    print(f"Raw data: data/{RAW_OUT} ({len(df)} posts)")
    print(f"Monthly data: data/{MONTHLY_OUT} ({len(monthly_df)} months)")
    
    return df, monthly_df

print("Main processing pipeline defined successfully")

Main processing pipeline defined successfully


In [8]:
# -----------------------------
# 5) Execute Analysis
# -----------------------------

# Your portfolio tickers
TICKERS = ["NVDA", "AMD", "ASML", "MU", "MRVL", "MSFT", "GOOGL", "AI", 
          "IONQ", "RGTI", "QBTS", "PLUG", "AEM", "VERU", "APP", 
          "ARBE", "INGM", "QQQ", "SPY", "RDDT", "PLTR"]

print("Starting PRAW-based Reddit sentiment analysis...")
print(f"Analyzing {len(TICKERS)} tickers")
print(f"Target subreddits per ticker: ~{len(CORE_TIER_A + CORE_TIER_B)} core + sector-specific")


# Run the analysis
raw_df, monthly_df = run_praw_analysis(TICKERS)

if monthly_df is not None:
    print(f"\n✅ Analysis completed successfully!")
    print(f"📊 Monthly sentiment data shape: {monthly_df.shape}")
    print(f"📈 Tickers with data: {monthly_df['ticker'].nunique()}")
    print(f"📅 Date range: {monthly_df['year'].min()}-{monthly_df['month'].min():02d} to {monthly_df['year'].max()}-{monthly_df['month'].max():02d}")
    
    # Show sample results
    print(f"\n🔍 Sample results:")
    print(monthly_df.head(10))
    
    # Show ticker coverage
    print(f"\n📋 Ticker coverage:")
    ticker_counts = monthly_df.groupby('ticker')['article_count'].sum().sort_values(ascending=False)
    print(ticker_counts)


Starting PRAW-based Reddit sentiment analysis...
Analyzing 21 tickers
Target subreddits per ticker: ~8 core + sector-specific
Connected to Reddit API as: Anonymous


Device set to use cpu


Using sentiment analyzer: finbert

=== Collecting data for NVDA from 12 subreddits ===
Found 6 relevant posts for NVDA in r/MachineLearning
Found 7 relevant posts for NVDA in r/SecurityAnalysis
Found 10 relevant posts for NVDA in r/StockMarket
Found 47 relevant posts for NVDA in r/ValueInvesting
Found 6 relevant posts for NVDA in r/artificial
Found 6 relevant posts for NVDA in r/dividends
Found 43 relevant posts for NVDA in r/investing
Found 6 relevant posts for NVDA in r/nvidia
Found 11 relevant posts for NVDA in r/options
Found 5 relevant posts for NVDA in r/singularity
Found 45 relevant posts for NVDA in r/stocks
Found 8 relevant posts for NVDA in r/wallstreetbets
Total posts collected for NVDA: 200

=== Collecting data for AMD from 11 subreddits ===
Found 11 relevant posts for AMD in r/AMD
Found 1 relevant posts for AMD in r/SecurityAnalysis
Found 6 relevant posts for AMD in r/StockMarket
Found 25 relevant posts for AMD in r/ValueInvesting
Found 8 relevant posts for AMD in r/builda

/Users/nataliamarko/miniconda3/envs/demand_forecast_env/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Sentiment analysis failed: list indices must be integers or slices, not str
Sentiment analysis failed: list indices must be integers or slices, not str
Sentiment analysis failed: list indices must be integers or slices, not str
Sentiment analysis failed: list indices must be integers or slices, not str
Sentiment analysis failed: list indices must be integers or slices, not str
Sentiment analysis failed: list indices must be integers or slices, not str
Sentiment analysis failed: list indices must be integers or slices, not str
Sentiment analysis failed: list indices must be integers or slices, not str
Sentiment analysis failed: list indices must be integers or slices, not str
Sentiment analysis failed: list indices must be integers or slices, not str
Sentiment analysis failed: list indices must be integers or slices, not str
Sentiment analysis failed: list indices must be integers or slices, not str
Sentiment analysis failed: list indices must be integers or slices, not str
Sentiment an

/var/folders/lj/jghsddqd5mxbhxdb6mwg7qcw0000gn/T/ipykernel_49566/1402493126.py:43: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['year_month'] = df['created_utc'].dt.to_period('M').dt.to_timestamp()
